In [15]:
import torch
from PIL import Image
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import os


class FrameSequenceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.data = df
        self.transform = transform

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        print(row['frame_paths'])
        frame_paths = row['frame_paths']  # List of paths
        label = int(row['label'])

        prev_frames = []
        for frame_path in frame_paths[:-1]:  # N previous frames
            image = Image.open(frame_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            prev_frames.append(image)

        chute_frame = Image.open(frame_paths[-1]).convert('RGB')
        if self.transform:
            chute_frame = self.transform(chute_frame)

        # prev_frames: list of N tensors → stacked
        prev_frames = torch.stack(prev_frames)  # (N, C, H, W)

        return prev_frames, chute_frame, label


In [16]:


def create_dataframe_for_frame_sequence(dataset_root):
    rows = []

    for shot_folder in os.listdir(dataset_root):
        folder_path = os.path.join(dataset_root, shot_folder)
        if not os.path.isdir(folder_path):
            continue

        label_path = os.path.join(folder_path, "label.txt")
        if not os.path.exists(label_path):
            print(f"Etiqueta no encontrada en {folder_path}")
            continue
        with open(label_path, "r") as f:
            label = int(f.read().strip())

        frame_paths = []
        for i in range(5):  # Los 5 frames previos
            prev_frame_path = os.path.join(folder_path, f"frame_prev_{i}.jpg")
            if os.path.exists(prev_frame_path):
                frame_paths.append(prev_frame_path)


        shot_frame_path = os.path.join(folder_path, "frame_shot.jpg")
        if os.path.exists(shot_frame_path):
            frame_paths.append(shot_frame_path)
        else:
            print(f"Frame del chute no encontrado en {folder_path}")
            continue

        rows.append([frame_paths, label])

    df = pd.DataFrame(rows, columns=["frame_paths", "label"])
    return df


dataset_path= "./Dataset_final/Frames_Bons_Definitius"
df = create_dataframe_for_frame_sequence(dataset_path)

obj=FrameSequenceDataset(df)
print(obj)

In [17]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
image_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

def get_dataloaders(csv_path, batch_size=32, split_ratio=0.8):
    dataset = FrameSequenceDataset(csv_path, transform=image_transforms['train'])

    train_size = int(split_ratio * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    # Replace transforms for validation set
    val_dataset.dataset.transform = image_transforms['val']

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader


In [18]:
import torch.nn as nn
import torch.nn.functional as F

class CNNLSTMClassifier(nn.Module):
    def __init__(self, cnn_out_dim=128, lstm_hidden=64):
        super(CNNLSTMClassifier, self).__init__()

        # CNN to extract features
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),  # (B, 3, 224, 224)
            nn.ReLU(),
            nn.MaxPool2d(2),                # (B, 16, 112, 112)
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                # (B, 32, 56, 56)
            nn.Flatten(),                   # (B, 32*56*56)
            nn.Linear(32 * 56 * 56, cnn_out_dim),
            nn.ReLU()
        )

        # LSTM for previous frame features
        self.lstm = nn.LSTM(input_size=cnn_out_dim, hidden_size=lstm_hidden, batch_first=True)

        # Final classification
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden + cnn_out_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, prev_frames, chute_frame):
        B, N, C, H, W = prev_frames.shape

        # Flatten temporal dimension to process through CNN
        prev_frames = prev_frames.view(B * N, C, H, W)
        prev_features = self.cnn(prev_frames)  # (B*N, cnn_out_dim)
        prev_features = prev_features.view(B, N, -1)  # (B, N, cnn_out_dim)

        # Process with LSTM
        _, (h_n, _) = self.lstm(prev_features)  # h_n: (1, B, lstm_hidden)
        lstm_out = h_n.squeeze(0)               # (B, lstm_hidden)

        # CNN feature for chute frame
        chute_feature = self.cnn(chute_frame)   # (B, cnn_out_dim)

        # Concatenate LSTM + chute
        combined = torch.cat([lstm_out, chute_feature], dim=1)  # (B, lstm_hidden + cnn_out_dim)
        output = self.classifier(combined)
        return output


In [31]:
def evaluate_cnn_lstm(model, val_loader, device, criterion):
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    with torch.no_grad():
        for prev_frames, chute_frame, labels in val_loader:
            prev_frames = prev_frames.to(device)
            chute_frame = chute_frame.to(device)
            #labels = labels.to(device).float()
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(prev_frames, chute_frame)
            #outputs=outputs.view(-1)
            loss=criterion(outputs,labels)
            val_loss += loss.item()*prev_frames.size(0)
            preds = (outputs > 0.5).int().squeeze()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss=val_loss/len(val_loader.dataset)
    accuracy= 100 * correct / total
    return accuracy, avg_loss


def train_cnn_lstm(model, train_loader, val_loader, epochs=10, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for prev_frames, chute_frame, labels in train_loader:
            prev_frames = prev_frames.to(device)
            chute_frame = chute_frame.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(prev_frames, chute_frame)
            #outputs=outputs.view(-1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * prev_frames.size(0)

        val_acc, val_loss = evaluate_cnn_lstm(model, val_loader, device,criterion)
        print(f"Epoch {epoch+1}, Training_Loss: {running_loss/len(train_loader.dataset):.4f},Validation_Loss:{val_loss:.4f} ,Val Acc: {val_acc:.2f}%")

    return model


In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
epoch=10
torch.manual_seed(0)
train_loader, val_loader = get_dataloaders(df, batch_size=32)
model=CNNLSTMClassifier(cnn_out_dim=128, lstm_hidden=64)
train_cnn_lstm(model, train_loader, val_loader, epochs=10, lr=1e-3)

cpu
['./Dataset_final/Frames_Bons_Definitius/2016-03-02_-_23-00_Levante_1_-_3_Real_Madrid_1_224p_shot_980/frame_prev_0.jpg', './Dataset_final/Frames_Bons_Definitius/2016-03-02_-_23-00_Levante_1_-_3_Real_Madrid_1_224p_shot_980/frame_prev_1.jpg', './Dataset_final/Frames_Bons_Definitius/2016-03-02_-_23-00_Levante_1_-_3_Real_Madrid_1_224p_shot_980/frame_prev_2.jpg', './Dataset_final/Frames_Bons_Definitius/2016-03-02_-_23-00_Levante_1_-_3_Real_Madrid_1_224p_shot_980/frame_prev_3.jpg', './Dataset_final/Frames_Bons_Definitius/2016-03-02_-_23-00_Levante_1_-_3_Real_Madrid_1_224p_shot_980/frame_prev_4.jpg', './Dataset_final/Frames_Bons_Definitius/2016-03-02_-_23-00_Levante_1_-_3_Real_Madrid_1_224p_shot_980/frame_shot.jpg']['./Dataset_final/Frames_Bons_Definitius/2014-12-09_-_22-45_Dortmund_1_-_1_Anderlecht_1_224p_shot_215/frame_prev_0.jpg', './Dataset_final/Frames_Bons_Definitius/2014-12-09_-_22-45_Dortmund_1_-_1_Anderlecht_1_224p_shot_215/frame_prev_1.jpg', './Dataset_final/Frames_Bons_Definiti

KeyboardInterrupt: 